# Session 1: Amazon Bedrock을 통한 Foundation Model 호출

### Index
1. Amazon Bedrock 소개
2. 사용 가능한 Model 조회
3. CLI를 통한 모델 호출
   - invoke-model vs converse 비교
4. SDK(boto3)를 통한 모델 호출
   - invoke_model
   - converse
   - Streaming (invoke_model_with_response_stream / converse_stream)
5. System Prompt & inferenceConfig

---
## Amazon Bedrock이란?

Amazon Bedrock은 AWS의 **완전 관리형 생성 AI 서비스**입니다.  
다양한 Foundation Model(FM)을 **API 하나로** 호출할 수 있으며, 인프라 관리 없이 바로 사용할 수 있습니다.

### 주요 특징
- **다양한 모델 선택**: Anthropic Claude, Amazon Nova, Meta Llama, Mistral 등
- **서버리스**: 프로비저닝 없이 API 호출만으로 사용
- **보안**: 데이터가 모델 학습에 사용되지 않음
- **통합**: AWS 서비스(IAM, CloudWatch, Lambda 등)와 자연스러운 연동

### 호출 방식 2가지

| 방식 | 설명 | 특징 |
|------|------|------|
| `invoke-model` / `invoke_model` | 모델별 고유 body 포맷 사용 | 모델마다 요청/응답 구조가 다름 |
| `converse` | 모델 무관 통일 인터페이스 | 모델을 바꿔도 코드 변경 불필요 **(권장)** |

> 💡 **왜 Converse가 권장인가?**  
> `invoke_model`은 Claude는 `anthropic_version` 필드가 필요하고, Nova는 `inferenceConfig` 구조가 다릅니다.  
> 모델을 교체할 때마다 body를 수정해야 합니다.  
> `converse`는 동일한 메시지 포맷으로 어떤 모델이든 호출할 수 있어 유지보수가 쉽습니다.

## 사용 가능한 Model 리스트 조회

In [ ]:
! aws bedrock list-foundation-models --region us-east-1 \
  --query 'modelSummaries[].[modelId, modelName, inferenceTypesSupported[0]]' \
  --output table

## CLI

> AWS CLI의 `bedrock-runtime` 서브커맨드를 사용합니다.  
> 아래에서 같은 "안녕" 메시지를 `invoke-model`과 `converse`로 각각 보내며 **body 포맷 차이**를 비교해보세요.

### Invoke Model

> ⚠️ 모델마다 body 포맷이 다릅니다.  
> - Nova: `messages[].content[].text` 구조 + `inferenceConfig`  
> - Claude: `messages[].content` (문자열) + `anthropic_version` 필수

In [ ]:
# Nova Lite 2
! aws bedrock-runtime invoke-model \
    --region us-east-1 \
    --model-id "us.amazon.nova-2-lite-v1:0" \
    --body '{"messages":[{"role":"user","content":[{"text":"안녕"}]}],"inferenceConfig":{"max_new_tokens":512}}' \
    --cli-binary-format raw-in-base64-out \
    /dev/stdout | jq .

In [ ]:
# Claude Sonnet 4.6
! aws bedrock-runtime invoke-model \
    --region us-east-1 \
    --model-id us.anthropic.claude-sonnet-4-6 \
    --body '{"anthropic_version":"bedrock-2023-05-31","max_tokens":1024,"messages":[{"role":"user","content":"안녕"}]}' \
    --cli-binary-format raw-in-base64-out \
    /dev/stdout | jq .

### Converse

> ✅ 모델에 관계없이 동일한 메시지 포맷을 사용합니다.  
> `modelId`만 바꾸면 다른 모델로 전환 가능 — 코드 수정 불필요!

In [ ]:
# Nova Lite 2
! aws bedrock-runtime converse \
    --region us-east-1 \
    --model-id us.amazon.nova-2-lite-v1:0 \
    --messages '[{"role":"user","content":[{"text":"안녕"}]}]'

In [ ]:
# Claude Sonnet 4.6
! aws bedrock-runtime converse \
    --region us-east-1 \
    --model-id "us.anthropic.claude-sonnet-4-6" \
    --messages '[{"role":"user","content":[{"text":"안녕"}]}]'

## SDK(boto3)

> Python에서 `boto3` 클라이언트를 사용하여 Bedrock 모델을 호출합니다.  
> CLI와 동일하게 `invoke_model`(모델별 포맷)과 `converse`(통일 포맷) 두 가지 방식이 있습니다.

In [ ]:
# ! python3.13 -m pip install boto3

In [ ]:
import boto3
import json

In [ ]:
client = boto3.client("bedrock-runtime", region_name="us-east-1")

### Invoke

> 모델별 고유 body를 JSON으로 직접 구성합니다.  
> 응답 파싱도 모델마다 다르다는 점에 주목하세요. (Claude: `content[0].text`, Nova: `output.message.content[0].text`)

In [ ]:
# Claude Sonnet 4.6
response = client.invoke_model(
  modelId="us.anthropic.claude-sonnet-4-6",
  body=json.dumps({
      "anthropic_version": "bedrock-2023-05-31",
      "max_tokens": 1024,
      "messages": [{"role": "user", "content": "안녕"}]
  })
)
print(json.loads(response["body"].read())["content"][0]["text"])

In [ ]:
# Nova 2 Lite
response = client.invoke_model(
  modelId="us.amazon.nova-2-lite-v1:0",
  body=json.dumps({
      "messages": [{"role": "user", "content": [{"text": "안녕"}]}],
      "inferenceConfig": {"maxTokens": 512}
  })
)
print(json.loads(response["body"].read())["output"]["message"]["content"][0]["text"])

### Converse

> ✅ 동일한 `messages` 포맷, 동일한 응답 구조.  
> `modelId`만 교체하면 됩니다. 프로덕션에서는 이 방식을 권장합니다.

In [ ]:
messages = [{"role": "user", "content": [{"text": "안녕"}]}]
nova_modelId = "us.amazon.nova-2-lite-v1:0"
claude_modelId = "us.anthropic.claude-sonnet-4-6"

In [ ]:
# Claude Sonnet 4.6
response = client.converse(
  modelId=claude_modelId,
  messages=messages
)
print(response["output"]["message"]["content"][0]["text"])

In [ ]:
# Nova 2 Lite
response = client.converse(
  modelId=nova_modelId,
  messages=messages
)
print(response["output"]["message"]["content"][0]["text"])

### Stream

> 긴 응답을 생성할 때 전체 완료를 기다리지 않고 **토큰 단위로 실시간 수신**합니다.  
> 챗봇 UX에서 필수적인 패턴입니다.
>
> | 방식 | API | 비고 |
> |------|-----|------|
> | invoke_model_with_response_stream | 모델별 body 포맷 | 모델별 스트림 파싱 다름 |
> | converse_stream | 통일 포맷 | 권장 |

In [ ]:
# ============================================================
# invoke_model_with_response_stream
# ============================================================
import boto3
import json

client = boto3.client("bedrock-runtime", region_name="us-east-1")

response = client.invoke_model_with_response_stream(
  modelId="us.amazon.nova-2-lite-v1:0",
  body=json.dumps({
      "schemaVersion": "messages-v1",
      "messages": [{"role": "user", "content": [{"text": "AWS Bedrock의 주요 기능을 3가지만 설명해줘."}]}],
      "inferenceConfig": {"maxTokens": 512}
  })
)

for event in response["body"]:
  chunk = json.loads(event["chunk"]["bytes"])
  if "contentBlockDelta" in chunk:
      print(chunk["contentBlockDelta"]["delta"]["text"], end="", flush=True)

### converse_stream (권장)

> ✅ 통일된 포맷으로 스트리밍. 모델 교체 시에도 파싱 코드 변경 불필요.

In [ ]:
# ============================================================
# converse_stream (권장)
# ============================================================

response = client.converse_stream(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "AWS Bedrock의 주요 기능을 3가지만 설명해줘."}]}]
)

for event in response["stream"]:
  if "contentBlockDelta" in event:
      print(event["contentBlockDelta"]["delta"]["text"], end="", flush=True)

## System Prompt & inferenceConfig

> 모델의 동작을 제어하는 핵심 파라미터입니다.  
> **System Prompt**로 모델의 역할과 규칙을 정의하고, **inferenceConfig**로 응답 생성 방식을 조절합니다.  
> 두 파라미터 모두 Converse API와 InvokeModel API에서 사용할 수 있습니다.

### System Prompt

> 모델에게 역할, 성격, 응답 규칙을 부여합니다.  
> `system` 파라미터에 리스트 형태로 전달합니다.

In [ ]:
# System Prompt 없이 호출
response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "AWS에 대해서 간단하게 설명해줘"}]}]
)
print("[System Prompt 없음]")
print(response["output"]["message"]["content"][0]["text"])

In [ ]:
# System Prompt 적용 — 5살 아이에게 설명하는 선생님 역할
response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  system=[{"text": "너는 5살 아이에게 설명하는 유치원 선생님이야. 쉬운 비유를 사용하고, 10문장 이내로 답해."}],
  messages=[{"role": "user", "content": [{"text": "AWS에 대해서 간단하게 설명해줘?"}]}]
)
print("[System Prompt 적용]")
print(response["output"]["message"]["content"][0]["text"])

### inferenceConfig

> 모델의 응답 생성 방식을 제어하는 파라미터입니다.
>
> | 파라미터 | 설명 | 범위 |
> |----------|------|------|
> | `maxTokens` | 최대 응답 토큰 수 | 1 ~ 모델 최대값 |
> | `temperature` | 높을수록 창의적, 낮을수록 일관적 | 0.0 ~ 1.0 |
> | `topP` | 누적 확률 기준 후보 토큰 범위 | 0.0 ~ 1.0 |

#### maxTokens 비교
> 응답 길이를 제한합니다. 짧게 설정하면 응답이 중간에 잘립니다.
>
> 주로 과도한 토큰 사용을 방지하기 위해서 사용합니다.

In [ ]:
# maxTokens = 20 (짧게 제한)
response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "AWS의 주요 서비스 5가지를 설명해줘"}]}],
  inferenceConfig={"maxTokens": 20}
)
print(f"[maxTokens=20] stopReason: {response['stopReason']}")
print(response["output"]["message"]["content"][0]["text"])

In [ ]:
# maxTokens = 500 (충분히)
response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "AWS의 주요 서비스 5가지를 설명해줘"}]}],
  inferenceConfig={"maxTokens": 500}
)
print(f"[maxTokens=500] stopReason: {response['stopReason']}")
print(response["output"]["message"]["content"][0]["text"])

#### temperature 비교
> 같은 질문을 temperature 0.0과 1.0으로 각각 호출하여 차이를 비교합니다.  
> - **0.0**: 가장 확률 높은 토큰만 선택 → 일관된 응답  
> - **1.0**: 다양한 토큰 선택 가능 → 창의적/랜덤한 응답
>
> 💡 여러 번 실행해보세요. temperature=0.0은 매번 거의 같은 답이 나오고, 1.0은 매번 다른 답이 나옵니다.

In [ ]:
# temperature = 0.0 (결정적) — 여러 번 실행해도 같은 결과

response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "1부터 50 사이 자연수 중 아무거나 딱 하나만, 숫자만 답해줘."}]}],
  inferenceConfig={"maxTokens": 200, "temperature": 0.0}
)
print("[temperature=0.0]")
print(response["output"]["message"]["content"][0]["text"])

In [ ]:
# temperature = 1.0 (창의적) — 실행할 때마다 다른 결과

response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "1부터 50 사이 자연수 중 아무거나 딱 하나만, 숫자만 답해줘."}]}],
  inferenceConfig={"maxTokens": 200, "temperature": 1.0}
)
print("[temperature=1.0]")
print(response["output"]["message"]["content"][0]["text"])

#### topP
> 누적 확률 상위 P%의 토큰만 후보로 사용합니다.  
> - **0.1**: 상위 10% 토큰만 → 매우 보수적  
> - **0.9**: 상위 90% 토큰 → 다양한 표현 가능
>
> 💡 일반적으로 temperature와 topP 중 하나만 조절합니다.  
> 둘 다 높이면 너무 랜덤해지고, 둘 다 낮추면 반복적인 응답이 됩니다.  
> 실무에서는 temperature만 조절하는 경우가 많습니다.

In [ ]:
# topP = 0.1 (보수적)

response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "'오늘 아침' 으로 시작하는 문장 하나 알려줘"}]}],
  inferenceConfig={"maxTokens": 200, "temperature": 1.0, "topP": 0.1}
)
print("[topP=0.1]")
print(response["output"]["message"]["content"][0]["text"])

In [ ]:
# topP = 1.0 (다양한)

response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "'오늘 아침' 으로 시작하는 문장 하나 알려줘"}]}],
  inferenceConfig={"maxTokens": 200, "temperature": 1.0, "topP": 1.0}
)
print("[topP=1.0]")
print(response["output"]["message"]["content"][0]["text"])